# **Summary of Exploration**

## **Automatic Validation Exploration Summary**

Please first see probe_autoval_rocauc_final.ipynb for the explanation of my automatic validation metric. 

For my exploration of what metric worked best (what worked, what didn't), see below. 

### **Initial Metric**
For my initial metric, I...

In [ ]:
# Initial activation validation score calculation
def compute_activation_validation_score(token_strs, scores, concept_text):
    """
    Sums the activation-validation scores for the whole text.
    Each token's contribution is: activation * abs(similarity of its group).
    The function groups tokens by separators and then sums the contributions.
    """
    total = 0.0
    current_group_tokens = []
    current_group_scores = []
    # Loop through tokens and accumulate group data
    for i, (token, score) in enumerate(zip(token_strs, scores)):
        current_group_tokens.append(token)
        current_group_scores.append(score)
        # when a separator is reached, finalise the group
        if is_group_separator(token_strs, i):
            group_string = "".join(current_group_tokens).strip()
            similarity = compute_semantic_similarity(group_string, concept_text)
            total += sum(sc * abs(similarity) for sc in current_group_scores)
            # Reset group lists for the next group
            current_group_tokens = []
            current_group_scores = []
    # Process any remaining tokens not ending with a seperator
    if current_group_tokens:
        group_string = "".join(current_group_tokens).strip()
        similarity = compute_semantic_similarity(group_string, concept_text)
        total += sum(sc * abs(similarity) for sc in current_group_scores)
    return total

fjdskldfjdskl

In [ ]:
# Pearson correlation calculation
def compute_activation_validation_score(token_strs, scores, concept_text):
    """
    Instead of summing activation*similarity, we compute a Pearson correlation
    between activation and similarity across all tokens (grouped by punctuation).
    
    Returns:
        float: The Pearson correlation (r) between each token's activation
               and its group's semantic similarity.
               If fewer than 2 tokens exist, returns 0.0 (arbitrary fallback).
    """
    # store activation and similarity for each token
    all_activations = []
    all_similarities = []
    
    current_group_tokens = []
    current_group_scores = []

    for i, (token, score) in enumerate(zip(token_strs, scores)):
        current_group_tokens.append(token)
        current_group_scores.append(score)
        
        # finalize the group a separator is reached
        if is_group_separator(token_strs, i):
            group_string = "".join(current_group_tokens).strip()
            similarity = compute_semantic_similarity(group_string, concept_text)
            
            # for each token in this group, activation is 'score',
            # similarity is 'similarity' (same for all tokens in the group).
            for sc in current_group_scores:
                all_activations.append(sc)
                all_similarities.append(similarity)
            
            # Reset group lists
            current_group_tokens = []
            current_group_scores = []
    
    # Process any remaining tokens not ended by a separator
    if current_group_tokens:
        group_string = "".join(current_group_tokens).strip()
        similarity = compute_semantic_similarity(group_string, concept_text)
        
        for sc in current_group_scores:
            all_activations.append(sc)
            all_similarities.append(similarity)
    
    # Compute pearson correlation over all tokens
    if len(all_activations) < 2:
        # Not enough points to compute correlation
        return 0.0
    
    r_value, p_value = pearsonr(all_activations, all_similarities)
    return r_value

### Final automatic validation?
See separate final notebook

## **New Hold-Out Test Set for Validation Summary**
### My approach
- Wanted to create some texts similar to the examples given - similar structure, extra validations
- Wanted to create example texts with more variety - medical texts, different structures, realistic still
- Wanted to create unrelated texts with more variety - focus on more than one person, colloqial vs formal, different topics, test the robustness of the concept detection   

Note: Can find simple explainations of each text in inputs/example_unrelated_description.txt   
This is for ease of use when evaluating using the automatic validation metric    
Contains information such as:   
- Structure of the text   
- Source of text   
- Is the text single-person focused? 
- What type of text - for unrelated    
- Other details: numbers, vocab, colloqial?


### Example texts
For example2, I prompted ChatGPT (o1) to generate similar, but different validation examples. I used the following prompt below:

*I am trying to generate validation text examples for some probes which detect medical specific concepts.     
The texts should be around 300 – 450 words in length. 
The texts should include clinical concepts as they would appear in physician notes, radiology reports, and medical diagnoses. Examples should use medical terminology and focus on clinical presentations, diagnostic findings, and treatment considerations*     

*An example is shown below...*    

*Please produce a different example text to the one above, that still stays within my requirements. The text can be structured differently, but needs to mimic a normal medical text.*    

*It is extremely vital that everything mentioned in the text is medically possible and that the text makes sense as a whole. The symptoms, medical histories, and laboratory results must all be viable, and the patient should present with a realistic set of conditions.*     

*The example text should include the following concepts, but in varying degrees. These concepts do not need to be fully named, however the concept itself should be clearly embedded within the example.*

*The concepts are listed below...*   

For example3, I used the same prompt, but asked to generate in a clinical report style.   

For example4, I wanted to look at including only specific concepts, to test if the probes behaved as expected. I wanted to make sure the probes were activating for specific concepts, not just medical text...    
*Can you give me a text, structured like the example I gave you, that only focuses on the following concepts (as well as other relating concepts): "hypothyroidism" "heavy alcohol use", "pregnancy”*

For example5, 6, I took from online case studies or textbooks.   
Sources: http://repository.stikesrspadgs.ac.id/71/1/100%20Case%20Studies%20in%20Pathophysiology-532hlm%20%28warna%20hanya%20cover%29.pdf, https://pmc.ncbi.nlm.nih.gov/articles/PMC7120678/   

## Unrelated texts
For unrelated2, I used o1 generation:
*Can you now create an unrelated text example. I would like the vague structure to mimic the structure of the example, in order to create a fair test. For the unrelated text example, do not mention or reference any of the concepts from the list before. The text should not be of the medical domain. An example unrelated text is shown below...*

I specified that I wanted an output with numbers/symbols:
- Found they were often activated on
- Similar to medical texts
- Use to validate probe specificity     

unrelated3.txt: Colloquial text from Reddit post, mis-spellings and lack of punctuation, first-person   

unrelated4.txt: Character analysis of fictional character, single-person focused, many names, rare vocab, literature focused    

unrelated5.txt: Scientific explanation of alcohol, chosen to validate if "heavy alcohol use" probe is word dependent or context dependent  

*"Sugar alcohols are alcohols that are made by hydrogenating sugar molecules. Hydrogenation basically means "sticking more hydrogen atoms onto it". Sugar alcohols are quite different from the types of alcohol that ethanol belongs to. For one thing, they are solid at room temperature (like sugar is). Also, you can't produce sugar alcohols through fermentation..."*

unrelated6.txt: Academic review article on corporate law, includes citations    

unrelated7.txt: Wikipedia article of famous person, single-person focused, currently not in use for validation   




## **Removing Bad Mined Negatives**


## **Positive/Negative Example Length**

## **Truncating Inputs**
Unsuccessful... 
Remember to go through ALL my files, and tell what didn't work
Show examples for this... - ss


## **Unanswered Questions + Future Directions**

Due to time restraints, I wasn't able to complete all the experiments I had planned or wanted to :( - sorry!   
So, below I've listed some unanswered questions/future directions - either from the ideas that you sent me to try out, or other ideas I had along the way that I didn't get to try. 

### Exploring 'Equivalent' Feature Difference Vectors
